# Amazon S3 → raw volume

Copies the POS Parquet object into the volume **byte for byte** — no Spark read,
no reserialisation. The file in the volume is identical to the file the store
systems produced, which is what makes the raw layer an audit trail.

Uses the Databricks Files API because Serverless will not copy from local `/tmp`
into a Unity Catalog volume.

In [ ]:
%pip install -q boto3==1.35.98 databricks-sdk

In [ ]:
import sys
from datetime import date
from pathlib import Path

for candidate in [Path.cwd(), *Path.cwd().parents]:
    if (candidate / "common_utils").is_dir():
        sys.path.insert(0, str(candidate))
        break

from common_utils.ingestors import copy_s3_to_volume, s3_client
from common_utils.logger import get_logger, log_info
from common_utils.observability import ensure_ops_schema, new_run_id, track
from common_utils.settings import get_secret, load_json, parse_run_date
from common_utils.writers import create_namespace

In [ ]:
dbutils.widgets.text("config_path", "ingestion/config/s3_sales.json")
dbutils.widgets.text("catalog", "retaildataplatform")
dbutils.widgets.text("bronze_schema", "bronze")
dbutils.widgets.text("raw_volume", "raw_data")
dbutils.widgets.text("secret_scope", "retail-platform-dev")
dbutils.widgets.text("run_date", date.today().isoformat())

config = load_json(dbutils.widgets.get("config_path"))
catalog = dbutils.widgets.get("catalog")
bronze_schema = dbutils.widgets.get("bronze_schema")
raw_volume = dbutils.widgets.get("raw_volume")
scope = dbutils.widgets.get("secret_scope")
run_date = parse_run_date(dbutils.widgets.get("run_date"))
run_id = new_run_id()

connection = config["connection"]
secrets = config["secret_keys"]
target = f"/Volumes/{catalog}/{bronze_schema}/{raw_volume}/{config['source_name']}/load_date={run_date}"
logger = get_logger("ingestion")

In [ ]:
create_namespace(spark, catalog, bronze_schema, raw_volume, comment="Bronze: raw landing volume and raw copies of source data")
ensure_ops_schema(spark, catalog)

with track(spark, catalog, run_id, run_date, task="s3_ingestion", layer="raw", entity=config["source_name"]) as stats:
    client = s3_client(
        access_key_id=get_secret(dbutils, scope, secrets["access_key_id"]),
        secret_access_key=get_secret(dbutils, scope, secrets["secret_access_key"]),
        region=connection["region"],
    )
    log_info(logger, "copying", uri=connection["uri"], target=target)

    # Replace the day's folder so a re-run cannot leave yesterday's file behind.
    try:
        dbutils.fs.rm(target, recurse=True)
    except Exception:  # noqa: BLE001 - folder may not exist on the first run
        pass

    written = copy_s3_to_volume(client, connection["uri"], target)
    stats.rows_written = len(written)
    log_info(logger, "landed", source=config["source_name"], files=len(written), path=target)

In [ ]:
display(spark.read.parquet(target).limit(10))